# Qwen3-VL LoRA Inference on Kaggle
## Invoice DocILE Extraction — Compact Multi-Pass Inference

This notebook runs **inference** using a fine-tuned `unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit` model with a LoRA adapter on Kaggle T4 GPU.

It is configured to avoid the long hallucination loops seen in the older single-pass setup by using a **compact four-pass JSON pipeline** and merging the results back into DocILE format.


### Before running:
1. **Upload your datasets to Kaggle** via _Notebook → Add Data_:
   - **Adapter dataset**: upload your `final_adapter/` folder contents → it will appear under `/kaggle/input/<your-adapter-dataset-slug>/`
   - **Invoice images dataset**: upload your `validation_image/` PNG files → `/kaggle/input/<your-images-dataset-slug>/`
   - **Ground-truth dataset** _(optional, for evaluation)_: upload your JSON files → `/kaggle/input/<your-gt-dataset-slug>/`
2. **Update the path variables** in the Configure Paths cell to match your dataset slugs.
3. **Enable GPU**: Notebook Settings → Accelerator → GPU T4 x2.

## Step 1: Install Dependencies

In [ ]:
# Install all required packages for Unsloth vision inference on Kaggle T4
# Removed %%capture to show progress
!pip install -q --upgrade \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "transformers>=4.45.0,<=5.3.0" \
    "peft>=0.18.0" \
    "trl>=0.15.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=1.0.0" \
    "qwen-vl-utils>=0.0.14" \
    "seaborn"

print("Installation complete.")

## Step 2: Configure Paths & Verify GPU

Update the dataset slug names below to match what you uploaded to Kaggle.

In [ ]:
import os
import torch

BASE_MODEL_ID = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"

# Update ADAPTER_PATH to the uploaded final_adapter directory for your Qwen3-VL LoRA.
ADAPTER_PATH = "/kaggle/input/datasets/user/single-template-invoices/final_adapter"
IMAGE_DIR = "/kaggle/input/datasets/user/single-template-invoices/validation_image"
JSON_DIR = "/kaggle/input/datasets/user/single-template-invoices/validation_groundtruth"
GEN_DIR = "/kaggle/working/document_gen"
DEBUG_DIR = "/kaggle/working/document_gen_debug"

os.makedirs(GEN_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

print(f"BASE_MODEL_ID: {BASE_MODEL_ID}")
print(f"ADAPTER_PATH : {ADAPTER_PATH}")
print(f"IMAGE_DIR    : {IMAGE_DIR}")
print(f"JSON_DIR     : {JSON_DIR}")
print(f"GEN_DIR      : {GEN_DIR}")
print(f"DEBUG_DIR    : {DEBUG_DIR}")
print()

# Verify CUDA GPU
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Enable GPU: Notebook Settings -> Accelerator -> GPU T4.")

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 3: Load the Fine-Tuned Qwen3-VL LoRA Adapter

This loads the `unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit` base model in 4-bit precision and attaches the new LoRA adapter using native Transformers + PEFT for more reliable inference on Kaggle.

In [ ]:
import gc
import os
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import PeftModel

if not os.path.exists(ADAPTER_PATH):
    raise FileNotFoundError(
        f"Adapter path not found: {ADAPTER_PATH}\n"
        "Upload the Qwen3-VL final_adapter folder to Kaggle and update ADAPTER_PATH in Step 2."
    )

print("Cleaning VRAM...")
gc.collect()
torch.cuda.empty_cache()

print(f"Loading base model: {BASE_MODEL_ID}")
print(f"Loading LoRA adapter from: {ADAPTER_PATH}")

processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    quantization_config=quant_config,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print("\nQwen3-VL base model + adapter loaded successfully.")
print(f"VRAM used after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 4: Compact Multi-Pass JSON-Safe Inference Setup

Instead of one long single-pass prompt that can loop for minutes and end as raw text, this setup uses **four compact passes**:

1. header / identity fields  
2. grand totals  
3. tax summary / VAT details  
4. line items  

The outputs are then merged back into a valid DocILE-style JSON locally.

In [ ]:
import gc
import json
import re
import time
from copy import deepcopy
from pathlib import Path

import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import StoppingCriteria, StoppingCriteriaList

INFERENCE_POLICY = "sliding_window_qwen3_compact_anti_loop_repair_v3_bbox_required"
IMAGE_MAX_PIXELS = 640 * 960
IMAGE_MAX_PIXELS_CANDIDATES = [IMAGE_MAX_PIXELS, 512 * 768, 384 * 640]

DOC_LEVEL_ALLOWED = {
    "document_id", "date_issue", "date_due", "purchase_order_id", "terms",
    "tax_name", "tax_amount", "sender_vat_id",
    "amount_due", "amount_total_tax", "amount_total_base",
    "recipient_name", "recipient_address", "recipient_delivery_name",
    "recipient_delivery_address", "sender_name", "sender_address", "vendor_phone",
}

LINE_LEVEL_ALLOWED = {
    "item_code", "item_description", "item_uom", "item_quantity",
    "item_amount", "item_amount_total",
}


def normalize_content_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(str(x) for x in content if x).strip()
    return str(content)


def extract_json_candidate(text: str) -> str:
    text = text.strip()
    if text.startswith("```json"):
        text = text[7:]
    if text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    text = text.strip()

    start = text.find("{")
    if start == -1:
        return text

    depth = 0
    in_string = False
    escape = False
    for index in range(start, len(text)):
        char = text[index]
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1]

    return text[start:]


def repair_json_candidate(text: str) -> str:
    repaired = text.strip()
    repaired = repaired.replace("“", '"').replace("”", '"').replace("’", "'")
    repaired = repaired.replace("\r", " ")
    repaired = re.sub(r",(\s*[}\]])", r"\1", repaired)
    return repaired


def _close_open_json_structures(text: str) -> str:
    text = text.rstrip()
    if not text:
        return text

    text = re.sub(r",\s*$", "", text)

    stack = []
    in_string = False
    escape = False
    last_safe_comma = None

    for idx, char in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            stack.append("}")
        elif char == "[":
            stack.append("]")
        elif char in "}]":
            if stack and char == stack[-1]:
                stack.pop()
        elif char == ",":
            last_safe_comma = idx

    if in_string:
        if last_safe_comma is not None:
            text = text[:last_safe_comma]
        else:
            text = re.sub(r',?\s*"[^"\\]*$', "", text)

    text = re.sub(r',?\s*"[^"\\]*"\s*:\s*$', "", text).rstrip()
    text = re.sub(r",(\s*[}\]])", r"\1", text)

    stack = []
    in_string = False
    escape = False
    for char in text:
        if in_string:
            if escape:
                escape = False
            elif char == "\\":
                escape = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            stack.append("}")
        elif char == "[":
            stack.append("]")
        elif char in "}]":
            if stack and char == stack[-1]:
                stack.pop()

    return text + "".join(reversed(stack))


def salvage_partial_json(text: str):
    candidate = repair_json_candidate(extract_json_candidate(text))
    working = candidate
    last_error = "unknown parse error"

    for _ in range(min(400, max(1, len(candidate)))):
        trial = _close_open_json_structures(working)
        try:
            return json.loads(trial), trial, "truncated_tail_repaired"
        except json.JSONDecodeError as exc:
            last_error = str(exc)
            if not working:
                break
            working = working[:-1].rstrip()
            working = re.sub(r",\s*$", "", working)

    return None, candidate, last_error


def try_parse_json(text: str):
    primary = extract_json_candidate(text)
    candidates = [primary, repair_json_candidate(primary)]
    seen = set()
    last_error = "unknown parse error"

    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        try:
            return json.loads(candidate), candidate, None
        except json.JSONDecodeError as exc:
            last_error = str(exc)

    salvaged, salvaged_text, salvage_note = salvage_partial_json(primary)
    if salvaged is not None:
        return salvaged, salvaged_text, salvage_note

    return None, candidates[-1], last_error


def _safe_page(page_value, default=1):
    try:
        if page_value is None or page_value == "":
            return default
        return int(page_value)
    except Exception:
        return default


def _clean_bbox_value(bbox):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return None
    cleaned = []
    for value in bbox:
        try:
            cleaned.append(float(value))
        except Exception:
            return None
    return cleaned


def _drop_entries_with_missing_bbox(payload):
    if not isinstance(payload, dict):
        return payload

    fields = payload.get("fields")
    if isinstance(fields, dict):
        cleaned_fields = {}
        for key, value in fields.items():
            if not isinstance(value, dict):
                continue
            text = value.get("text")
            bbox = _clean_bbox_value(value.get("bbox"))
            if text not in (None, "", [], {}) and bbox is not None:
                cleaned_fields[key] = {
                    "text": text,
                    "bbox": bbox,
                    "page": _safe_page(value.get("page", 1), 1),
                }
        payload["fields"] = cleaned_fields

    tax_rows = payload.get("tax_rows")
    if isinstance(tax_rows, list):
        cleaned_rows = []
        for row in tax_rows:
            if not isinstance(row, dict):
                continue
            cleaned_row = {}
            for key, value in row.items():
                if not isinstance(value, dict):
                    continue
                text = value.get("text")
                bbox = _clean_bbox_value(value.get("bbox"))
                if text not in (None, "", [], {}) and bbox is not None:
                    cleaned_row[key] = {
                        "text": text,
                        "bbox": bbox,
                        "page": _safe_page(value.get("page", 1), 1),
                    }
            if cleaned_row:
                cleaned_rows.append(cleaned_row)
        payload["tax_rows"] = cleaned_rows

    line_items = payload.get("line_items")
    if isinstance(line_items, list):
        cleaned_items = []
        for row_idx, row in enumerate(line_items):
            if not isinstance(row, dict):
                continue
            row_fields = row.get("fields", {})
            if not isinstance(row_fields, dict):
                continue
            cleaned_row_fields = {}
            for key, value in row_fields.items():
                if not isinstance(value, dict):
                    continue
                text = value.get("text")
                bbox = _clean_bbox_value(value.get("bbox"))
                if text not in (None, "", [], {}) and bbox is not None:
                    cleaned_row_fields[key] = {
                        "text": text,
                        "bbox": bbox,
                        "page": _safe_page(value.get("page", row.get("page", 1)), _safe_page(row.get("page", 1), 1)),
                    }
            if cleaned_row_fields:
                cleaned_items.append({
                    "line_item_id": row.get("line_item_id", row_idx),
                    "page": _safe_page(row.get("page", 1), 1),
                    "fields": cleaned_row_fields,
                })
        payload["line_items"] = cleaned_items

    return payload


def _docile_entry(fieldtype, text=None, bbox=None, page=1, line_item_id=None):
    return {
        "bbox": bbox if isinstance(bbox, list) and len(bbox) == 4 else None,
        "page": _safe_page(page, 1),
        "score": None,
        "text": text,
        "fieldtype": fieldtype,
        "line_item_id": line_item_id,
        "use_only_for_ap": False,
    }


class RepetitionStoppingCriteria(StoppingCriteria):
    """Stop generation early if the tail starts looping heavily."""

    def __init__(self, tokenizer, prompt_length: int, window: int = 48, repeats: int = 3, check_after: int = 96):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length
        self.window = window
        self.repeats = repeats
        self.check_after = check_after

    def __call__(self, input_ids, _scores, **_kwargs):
        try:
            generated = input_ids[0][self.prompt_length:].tolist()
        except Exception:
            return False

        if len(generated) < max(self.check_after, self.window * self.repeats):
            return False

        tail = generated[-self.window * self.repeats:]
        chunks = [tail[i * self.window:(i + 1) * self.window] for i in range(self.repeats)]
        if len(chunks) >= 3 and chunks[-1] == chunks[-2] == chunks[-3]:
            print("⚠ Anti-loop stop triggered: repeated token block detected.")
            return True

        return False


def _compact_fields_to_docile(payload: dict):
    extractions = []
    source = payload.get("fields", payload.get("field_extractions", {}))

    items = []
    if isinstance(source, dict):
        items.extend(list(source.items()))
    elif isinstance(source, list):
        for entry in source:
            if isinstance(entry, dict) and entry.get("fieldtype"):
                items.append((entry.get("fieldtype"), entry))

    tax_rows = payload.get("tax_rows", [])
    if isinstance(tax_rows, list):
        for row in tax_rows:
            if not isinstance(row, dict):
                continue
            for fieldtype, value in row.items():
                if fieldtype in DOC_LEVEL_ALLOWED:
                    items.append((fieldtype, value))

    seen = set()
    for fieldtype, value in items:
        if fieldtype not in DOC_LEVEL_ALLOWED:
            continue

        if isinstance(value, dict):
            text = value.get("text")
            bbox = value.get("bbox")
            page = value.get("page", 1)
        else:
            text = value
            bbox = None
            page = 1

        if text in (None, "", [], {}):
            continue

        dedupe_key = (fieldtype, str(text).strip(), tuple(bbox) if isinstance(bbox, list) else None)
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)

        extractions.append(
            _docile_entry(
                fieldtype=fieldtype,
                text=text,
                bbox=bbox,
                page=page,
                line_item_id=None,
            )
        )

    return extractions


def _compact_line_items_to_docile(payload: dict):
    extractions = []
    seen = set()

    source = payload.get("line_items", [])
    if isinstance(source, list):
        for row_idx, row in enumerate(source):
            if not isinstance(row, dict):
                continue

            line_item_id = row.get("line_item_id", row_idx)
            row_page = _safe_page(row.get("page", 1), 1)
            row_fields = row.get("fields", {})

            if isinstance(row_fields, dict):
                for fieldtype, value in row_fields.items():
                    if fieldtype not in LINE_LEVEL_ALLOWED:
                        continue

                    if isinstance(value, dict):
                        text = value.get("text")
                        bbox = value.get("bbox")
                        page = value.get("page", row_page)
                    else:
                        text = value
                        bbox = None
                        page = row_page

                    if text in (None, "", [], {}):
                        continue

                    dedupe_key = (line_item_id, fieldtype, str(text).strip(), tuple(bbox) if isinstance(bbox, list) else None)
                    if dedupe_key in seen:
                        continue
                    seen.add(dedupe_key)

                    extractions.append(
                        _docile_entry(
                            fieldtype=fieldtype,
                            text=text,
                            bbox=bbox,
                            page=page,
                            line_item_id=line_item_id,
                        )
                    )

    return extractions


def _merge_metadata(*parts):
    merged = {
        "cluster_id": None,
        "currency": None,
        "document_type": None,
        "language": None,
        "original_filename": None,
        "page_count": None,
        "page_sizes_at_200dpi": None,
        "page_to_table_grid": {},
        "rossum_annotation_url": None,
        "rossum_status": None,
        "rossum_arrived_at": None,
        "rossum_exported_at": None,
        "rossum_export_failed_at": None,
        "rossum_modified_at": None,
        "rossum_modifier": None,
        "rossum_schema_url": None,
    }

    for part in parts:
        meta = part.get("metadata", {})
        if not isinstance(meta, dict):
            continue
        for key, value in meta.items():
            if value not in (None, "", [], {}):
                merged[key] = value

    return merged


def _merge_docile_parts(image_path: str, identity_part: dict, amount_part: dict, tax_part: dict, line_part: dict):
    field_extractions = []
    field_extractions.extend(_compact_fields_to_docile(identity_part))
    field_extractions.extend(_compact_fields_to_docile(amount_part))
    field_extractions.extend(_compact_fields_to_docile(tax_part))

    line_item_extractions = _compact_line_items_to_docile(line_part)
    line_item_headers = line_part.get("line_item_headers", []) if isinstance(line_part.get("line_item_headers", []), list) else []

    metadata = _merge_metadata(identity_part, amount_part, tax_part, line_part)
    if not metadata.get("original_filename"):
        metadata["original_filename"] = Path(image_path).name

    parts = [identity_part, amount_part, tax_part, line_part]
    total_time = round(sum(float(part.get("processing_time_s", 0) or 0) for part in parts), 2)
    total_prompt = sum((part.get("token_usage", {}) or {}).get("prompt_tokens") or 0 for part in parts)
    total_completion = sum((part.get("token_usage", {}) or {}).get("completion_tokens") or 0 for part in parts)
    total_tokens = sum((part.get("token_usage", {}) or {}).get("total_tokens") or 0 for part in parts)

    return {
        "field_extractions": field_extractions,
        "line_item_extractions": line_item_extractions,
        "line_item_headers": line_item_headers,
        "metadata": metadata,
        "processing_time_s": total_time,
        "model_used": BASE_MODEL_ID,
        "finish_reason": "windowed_merge_compact",
        "token_usage": {
            "prompt_tokens": total_prompt,
            "completion_tokens": total_completion,
            "total_tokens": total_tokens,
            "passes": {
                "identity": identity_part.get("token_usage", {}),
                "amounts": amount_part.get("token_usage", {}),
                "tax_details": tax_part.get("token_usage", {}),
                "line_items": line_part.get("token_usage", {}),
            },
        },
    }


COMPACT_RULES = """Return exactly one compact JSON object and nothing else. Use only the allowed visible keys. Recover every allowed field or row cell that is visibly present in the invoice with best effort. For every predicted text value that is visibly present, also predict its bbox and page; do not leave bbox null for visible extracted text. If a value has no reliable bbox, omit that field instead of returning a null bbox. Do not leave a visibly present allowed field blank and do not omit it unless it is truly absent or unreadable. No markdown, no explanation, no null placeholders, and no duplicated rows. bbox must be [x1, y1, x2, y2], and page is 1-based. Keep metadata-only values such as document_type, currency, and language under metadata."""

IDENTITY_PROMPT = f"""{COMPACT_RULES}

Task: extract only header / identity / contact fields in compact DocILE-style JSON.
Allowed fields: document_id, date_issue, date_due, purchase_order_id, terms, sender_name, sender_address, recipient_name, recipient_address, recipient_delivery_name, recipient_delivery_address, vendor_phone.

Return this shape:
{{
  "fields": {{
    "<visible_field_name>": {{"text": "...", "bbox": [x1, y1, x2, y2], "page": 1}}
  }},
  "metadata": {{"document_type": "...", "currency": "...", "language": "...", "original_filename": "...", "page_count": 1}}
}}
Only include keys that are clearly visible with both text and bbox. Omit missing fields instead of using null bbox.
"""

AMOUNTS_PROMPT = f"""{COMPACT_RULES}

Task: extract only the grand totals / balance-summary region.
Allowed fields: amount_due, amount_total_tax, amount_total_base.
Do not include tax rows or line items.

Return this shape:
{{
  "fields": {{
    "<visible_total_field>": {{"text": "...", "bbox": [x1, y1, x2, y2], "page": 1}}
  }},
  "metadata": {{"currency": "...", "document_type": "..."}}
}}
Only include totals that have both visible text and bbox.
"""

TAX_DETAILS_PROMPT = f"""{COMPACT_RULES}

Task: extract only the tax / VAT summary block.
Allowed fields: tax_name, tax_amount, sender_vat_id.
Do not include grand totals or line-item fields.

Return this shape:
{{
  "tax_rows": [
    {{
      "<visible_tax_field>": {{"text": "...", "bbox": [x1, y1, x2, y2], "page": 1}}
    }}
  ],
  "metadata": {{"currency": "...", "document_type": "..."}}
}}
Each tax row may contain `tax_name`, `tax_amount`, and `sender_vat_id` when visible. Omit missing keys and do not repeat the same row twice. Do not return tax entries with null bbox.
"""

LINE_ITEMS_PROMPT = f"""{COMPACT_RULES}

Task: extract only invoice line items.
Allowed fields: item_code, item_description, item_uom, item_quantity, item_amount, item_amount_total.

Return this shape:
{{
  "line_items": [
    {{
      "line_item_id": 0,
      "page": 1,
      "fields": {{"<visible_line_item_field>": {{"text": "...", "bbox": [x1, y1, x2, y2]}}}}
    }}
  ],
  "line_item_headers": []
}}
Use one object per visible row. Include only visible fields with non-null bbox and stop once all visible rows are covered.
"""


def build_messages(img_path: str, prompt: str, max_pixels: int = IMAGE_MAX_PIXELS) -> list:
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path, "max_pixels": max_pixels},
                {"type": "text", "text": prompt},
            ],
        }
    ]


def _generate_json_pass(image_path: str, prompt: str, max_new_tokens: int, label: str, decode_mode: str = "greedy"):
    stop_tokens = [processor.tokenizer.eos_token_id]
    vocab = processor.tokenizer.get_vocab()
    if "<|im_end|>" in vocab:
        stop_tokens.append(processor.tokenizer.convert_tokens_to_ids("<|im_end|>"))

    last_error = None
    for current_max_pixels in IMAGE_MAX_PIXELS_CANDIDATES:
        gc.collect()
        torch.cuda.empty_cache()

        try:
            messages = build_messages(image_path, prompt, max_pixels=current_max_pixels)
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info([messages])
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )

            model_device = next(model.parameters()).device
            inputs = {
                key: value.to(model_device) if torch.is_tensor(value) else value
                for key, value in inputs.items()
            }

            prompt_tokens = int(inputs["input_ids"].shape[1])
            stopping = StoppingCriteriaList([
                RepetitionStoppingCriteria(
                    tokenizer=processor.tokenizer,
                    prompt_length=prompt_tokens,
                    window=48,
                    repeats=3,
                    check_after=min(128, max_new_tokens),
                )
            ])

            generation_config = deepcopy(model.generation_config)
            generation_config.max_new_tokens = max_new_tokens
            generation_config.min_new_tokens = 0
            generation_config.use_cache = True
            generation_config.eos_token_id = stop_tokens
            generation_config.pad_token_id = processor.tokenizer.pad_token_id
            generation_config.repetition_penalty = 1.12
            generation_config.renormalize_logits = True

            if decode_mode == "greedy":
                generation_config.do_sample = False
                generation_config.temperature = 1.0
                generation_config.top_p = 1.0
                generation_config.top_k = 50
            else:
                generation_config.do_sample = True
                generation_config.temperature = 0.12
                generation_config.top_p = 0.85
                generation_config.top_k = 20

            t0 = time.time()
            with torch.inference_mode():
                generated_ids = model.generate(
                    **inputs,
                    generation_config=generation_config,
                    stopping_criteria=stopping,
                )
            elapsed = time.time() - t0

            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
            ]
            completion_tokens = int(len(generated_ids_trimmed[0]))
            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )[0]

            parsed, cleaned, parse_note = try_parse_json(normalize_content_text(output_text))
            if parsed is None:
                if completion_tokens >= max_new_tokens:
                    print(f"⚠ {label} reached max_new_tokens={max_new_tokens}; attempting truncated JSON recovery failed.")
                return {
                    "error": f"{label} pass did not return valid JSON",
                    "parse_error": parse_note,
                    "raw_output_preview": output_text[:1600],
                    "cleaned_output_preview": cleaned[:1600],
                    "processing_time_s": round(elapsed, 2),
                    "token_usage": {
                        "prompt_tokens": prompt_tokens,
                        "completion_tokens": completion_tokens,
                        "total_tokens": prompt_tokens + completion_tokens,
                    },
                    "max_pixels": current_max_pixels,
                }

            parsed = _drop_entries_with_missing_bbox(parsed)

            if completion_tokens >= max_new_tokens:
                print(f"⚠ {label} hit max_new_tokens={max_new_tokens}, but the JSON was repaired successfully.")
            if parse_note:
                parsed["parse_repair"] = parse_note

            parsed["processing_time_s"] = round(elapsed, 2)
            parsed["token_usage"] = {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": prompt_tokens + completion_tokens,
            }
            parsed["max_pixels"] = current_max_pixels
            return parsed

        except torch.cuda.OutOfMemoryError as error:
            last_error = error
            print(f"OOM during {label} at max_pixels={current_max_pixels}. Retrying with a lower cap...")
            gc.collect()
            torch.cuda.empty_cache()
            continue
        except Exception as error:
            return {
                "error": f"{label} pass failed",
                "details": str(error),
                "max_pixels": current_max_pixels,
            }

    return {
        "error": f"{label} pass failed after exhausting pixel retries",
        "details": str(last_error),
    }


def run_vision_inference(image_path: str, decode_mode: str = "greedy") -> dict:
    identity_result = _generate_json_pass(image_path, IDENTITY_PROMPT, max_new_tokens=640, label="identity", decode_mode=decode_mode)
    if "error" in identity_result:
        return {"error": "Sliding-window identity pass failed", "details": identity_result}

    amount_result = _generate_json_pass(image_path, AMOUNTS_PROMPT, max_new_tokens=320, label="amounts", decode_mode=decode_mode)
    if "error" in amount_result:
        return {"error": "Sliding-window amount pass failed", "details": amount_result}

    tax_result = _generate_json_pass(image_path, TAX_DETAILS_PROMPT, max_new_tokens=384, label="tax_details", decode_mode=decode_mode)
    if "error" in tax_result:
        return {"error": "Sliding-window tax-details pass failed", "details": tax_result}

    line_result = _generate_json_pass(image_path, LINE_ITEMS_PROMPT, max_new_tokens=1400, label="line_items", decode_mode=decode_mode)
    if "error" in line_result:
        return {"error": "Sliding-window line-item pass failed", "details": line_result}

    return _merge_docile_parts(image_path, identity_result, amount_result, tax_result, line_result)


print(f"Inference policy: {INFERENCE_POLICY}")
print(f"IMAGE_MAX_PIXELS={IMAGE_MAX_PIXELS}")
print(f"Pixel retry caps: {IMAGE_MAX_PIXELS_CANDIDATES}")
print("Anti-loop compact 4-pass inference with bbox-required extraction is ready for Qwen3-VL.")

## Step 4.5: Visualization & OCR Utilities (Standalone Helpers)

In [ ]:
import json
import random
import re
import subprocess
import sys
from copy import deepcopy
from difflib import SequenceMatcher
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

VIS_GROUND_TRUTH_DIR = Path(JSON_DIR)
VIS_OUTPUT_DIR = Path(GEN_DIR) / "overlay_output_with_fieldnames"
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATH_DIR = Path(IMAGE_DIR)
RAW_PREDICTION_PATH_DIR = Path(GEN_DIR)
REFINED_PREDICTION_PATH_DIR = RAW_PREDICTION_PATH_DIR / "refined_ocr"
REFINED_PREDICTION_PATH_DIR.mkdir(parents=True, exist_ok=True)

# OCR refinement switches
ENABLE_OCR_REFINEMENT = True
FORCE_RERUN_OCR_REFINEMENT = False
OCR_REFINEMENT_MIN_SCORE = 0.72

try:
    overlay_font = ImageFont.truetype("arial.ttf", 16)
except Exception:
    overlay_font = ImageFont.load_default()

RapidOCR = None
if ENABLE_OCR_REFINEMENT:
    try:
        from rapidocr_onnxruntime import RapidOCR
    except ImportError:
        print("Installing rapidocr-onnxruntime for OCR-based bbox refinement...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidocr-onnxruntime"], check=False)
        try:
            from rapidocr_onnxruntime import RapidOCR
        except Exception as exc:
            RapidOCR = None
            print(f"⚠ OCR refinement package unavailable: {exc}")
else:
    print("OCR refinement is disabled. Using raw predicted bboxes only.")

OCR_ENGINE = RapidOCR() if (ENABLE_OCR_REFINEMENT and RapidOCR is not None) else None
_OCR_CACHE = {}


def normalize_match_text(text):
    if text is None:
        return ""
    cleaned = str(text).strip().lower()
    cleaned = cleaned.replace("$", "")
    cleaned = cleaned.replace(",", "")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def compact_match_text(text):
    return re.sub(r"[^a-z0-9./%:-]+", "", normalize_match_text(text))


def polygon_to_bbox(points):
    if not points or len(points) < 4:
        return None
    try:
        xs = [float(pt[0]) for pt in points]
        ys = [float(pt[1]) for pt in points]
    except Exception:
        return None
    return [round(min(xs), 1), round(min(ys), 1), round(max(xs), 1), round(max(ys), 1)]


def bbox_center(bbox):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
    except Exception:
        return None
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def bbox_distance_score(box_a, box_b):
    center_a = bbox_center(box_a)
    center_b = bbox_center(box_b)
    if center_a is None or center_b is None:
        return 0.5
    dx = center_a[0] - center_b[0]
    dy = center_a[1] - center_b[1]
    dist = (dx ** 2 + dy ** 2) ** 0.5
    return max(0.0, 1.0 - (dist / 1500.0))


def load_ocr_entries(image_path):
    cache_key = str(image_path)
    if cache_key in _OCR_CACHE:
        return _OCR_CACHE[cache_key]

    entries = []
    if OCR_ENGINE is None or not Path(image_path).exists():
        _OCR_CACHE[cache_key] = entries
        return entries

    try:
        ocr_result, _ = OCR_ENGINE(str(image_path))
    except Exception as exc:
        print(f"⚠ OCR failed on {Path(image_path).name}: {exc}")
        _OCR_CACHE[cache_key] = entries
        return entries

    for item in ocr_result or []:
        if not isinstance(item, (list, tuple)) or len(item) < 3:
            continue
        points, text, score = item[0], item[1], item[2]
        bbox = polygon_to_bbox(points)
        if not text or bbox is None:
            continue
        try:
            score = float(score)
        except Exception:
            score = 0.0
        entries.append({
            "text": str(text),
            "norm": normalize_match_text(text),
            "compact": compact_match_text(text),
            "bbox": bbox,
            "score": score,
        })

    _OCR_CACHE[cache_key] = entries
    return entries


def find_best_ocr_match(target_text, ocr_entries, current_bbox=None):
    target_norm = normalize_match_text(target_text)
    target_compact = compact_match_text(target_text)
    if not target_compact:
        return None

    best_entry = None
    best_score = 0.0

    for entry in ocr_entries:
        norm = entry["norm"]
        compact = entry["compact"]
        if not compact:
            continue

        exact_score = 1.0 if target_compact == compact else 0.0
        contains_score = 0.0
        if target_compact in compact or compact in target_compact:
            contains_score = min(len(target_compact), len(compact)) / max(len(target_compact), len(compact))

        fuzzy_score = max(
            SequenceMatcher(None, target_norm, norm).ratio(),
            SequenceMatcher(None, target_compact, compact).ratio(),
        )
        text_score = max(exact_score, contains_score, fuzzy_score)
        location_score = bbox_distance_score(current_bbox, entry["bbox"])
        combined_score = (0.82 * text_score) + (0.18 * location_score)
        combined_score *= 0.9 + (0.1 * max(0.0, min(entry["score"], 1.0)))

        if combined_score > best_score:
            best_score = combined_score
            best_entry = entry

    min_required = OCR_REFINEMENT_MIN_SCORE if len(target_compact) >= 4 else 0.88
    if best_entry is None or best_score < min_required:
        return None

    return {
        "bbox": best_entry["bbox"],
        "ocr_text": best_entry["text"],
        "score": round(best_score, 4),
    }


def refine_docile_bboxes(pred_data, image_path):
    refined = deepcopy(pred_data)
    ocr_entries = load_ocr_entries(image_path)
    if not ocr_entries:
        return refined, {"candidate_boxes": 0, "refined_boxes": 0}

    candidate_boxes = 0
    refined_boxes = 0

    for section_name in ("field_extractions", "line_item_extractions"):
        for item in refined.get(section_name, []):
            text_value = item.get("text")
            if not text_value:
                continue

            candidate_boxes += 1
            original_bbox = item.get("bbox")
            match = find_best_ocr_match(text_value, ocr_entries, current_bbox=original_bbox)
            if match is None:
                continue

            if original_bbox != match["bbox"]:
                item["_bbox_model_raw"] = original_bbox
                item["bbox"] = match["bbox"]
                item["_ocr_match_text"] = match["ocr_text"]
                item["_ocr_match_score"] = match["score"]
                refined_boxes += 1

    meta = refined.setdefault("__metadata__", {})
    meta["ocr_bbox_refinement"] = {
        "enabled": ENABLE_OCR_REFINEMENT,
        "candidate_boxes": candidate_boxes,
        "refined_boxes": refined_boxes,
        "engine": "RapidOCR" if OCR_ENGINE is not None else "unavailable",
    }
    return refined, meta["ocr_bbox_refinement"]


def draw_bbox_and_label(draw, bbox, label, color, width=2):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return
    x1, y1, x2, y2 = bbox
    draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
    label_y = max(y1 - 18, 0)
    draw.text((x1, label_y), label, fill=color, font=overlay_font)


def collect_overlay_items(doc_json, prefix, color):
    items = []
    for entry in doc_json.get("field_extractions", []) + doc_json.get("line_item_extractions", []):
        bbox = entry.get("bbox")
        if not bbox or len(bbox) != 4:
            continue
        page = entry.get("page")
        field_type = entry.get("fieldtype", "unknown")
        line_item_id = entry.get("line_item_id")
        suffix = f"#{line_item_id}" if line_item_id is not None else ""
        items.append({
            "page": page,
            "bbox": bbox,
            "label": f"{prefix}:{field_type}{suffix}",
            "color": color,
        })
    return items




## Step 5: Run Anti-Loop Compact Inference on Images

This cell now:

- **clears old outputs** under `/kaggle/working` to refresh JSON, debug, OCR, and overlay results
- runs inference on **5 random invoice images by default**
- still supports `TEST_IMAGE_NAME` if you want one specific file
- keeps the compact four-pass anti-loop safeguards for `Qwen3-VL`

In [ ]:
import gc
import json
import os
import random
import shutil

import pandas as pd
import torch
from IPython.display import display
from PIL import Image

if not os.path.exists(IMAGE_DIR):
    raise FileNotFoundError(f"IMAGE_DIR not found: {IMAGE_DIR}\nCheck the dataset slug in Step 2.")

valid_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(".png")])
if not valid_files:
    raise FileNotFoundError(f"No PNG files found in {IMAGE_DIR}")

# Inference controls
process_all = False
TEST_IMAGE_NAME = None   # e.g. "invoice_sample.png"
NUM_RANDOM_SAMPLES = 5
decode_mode = "greedy"  # safest default with anti-loop guard now enabled
CLEAN_OLD_OUTPUTS = True

if TEST_IMAGE_NAME is None:
    random.seed(None)
else:
    seed = 3407
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def refresh_output_dirs(*paths):
    refreshed = []
    for path in paths:
        if os.path.exists(path):
            shutil.rmtree(path)
        os.makedirs(path, exist_ok=True)
        refreshed.append(path)
    return refreshed


if CLEAN_OLD_OUTPUTS:
    refreshed_dirs = refresh_output_dirs(GEN_DIR, DEBUG_DIR)
    print("Refreshed working output folders:")
    for folder in refreshed_dirs:
        print(f"  - {folder}")
    print()

if process_all:
    target_files = valid_files
    print(f"Found {len(valid_files)} images. Starting anti-loop compact inference on all files...\n")
else:
    if TEST_IMAGE_NAME and TEST_IMAGE_NAME in valid_files:
        target_files = [TEST_IMAGE_NAME]
        print(f"Selected requested image for inference: {target_files[0]}\n")
    else:
        if TEST_IMAGE_NAME:
            print(f"⚠ Requested file not found, falling back to a random sample set: {TEST_IMAGE_NAME}")
        sample_count = min(NUM_RANDOM_SAMPLES, len(valid_files))
        target_files = random.sample(valid_files, k=sample_count)
        print(f"Selected {len(target_files)} random images for inference:")
        for idx, name in enumerate(target_files, start=1):
            print(f"  {idx}. {name}")
        print()

print(f"Inference policy : {INFERENCE_POLICY}")
print(f"Decode mode      : {decode_mode}")
print(f"Pixel retry caps : {IMAGE_MAX_PIXELS_CANDIDATES}")
print(f"{'File':<45} {'Time (s)':>8} {'Tokens':>7} {'Status'}")
print("-" * 80)

results_log = []

for filename in target_files:
    base_name = os.path.splitext(filename)[0]
    img_path = os.path.join(IMAGE_DIR, filename)
    gen_json_path = os.path.join(GEN_DIR, base_name + ".json")
    debug_json_path = os.path.join(DEBUG_DIR, base_name + "_error.json")

    with Image.open(img_path) as image:
        width, height = image.size
    print(f"Input image size: {width}x{height} px")

    gc.collect()
    torch.cuda.empty_cache()

    result = run_vision_inference(img_path, decode_mode=decode_mode)

    if "error" in result:
        status = "ERROR"
        detail = result.get("details", {}) if isinstance(result.get("details"), dict) else {}
        processing_time = detail.get("processing_time_s", result.get("processing_time_s", "?"))
        token_usage = detail.get("token_usage", result.get("token_usage", {})) if isinstance(detail, dict) else {}
        total_tokens = token_usage.get("total_tokens", "?") if isinstance(token_usage, dict) else "?"

        with open(debug_json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)

        print(f"❌ Error: {result['error']}")
        if isinstance(detail, dict):
            if detail.get("parse_error"):
                print(f"   Parse error: {detail['parse_error']}")
            if detail.get("details"):
                print(f"   Details: {detail['details']}")
            if detail.get("raw_output_preview"):
                print("\nRaw output preview:")
                print(detail["raw_output_preview"])
        print(f"   Saved debug file: {debug_json_path}")
    else:
        status = "OK"
        with open(gen_json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)

        processing_time = result.get("processing_time_s", "?")
        token_usage = result.get("token_usage", {})
        total_tokens = token_usage.get("total_tokens", "?") if isinstance(token_usage, dict) else "?"

        print(f"✓ JSON saved to {gen_json_path}")
        print(f"  Finish reason: {result.get('finish_reason')}")
        print(f"  Total tokens : {total_tokens}")

        pass_usage = token_usage.get("passes", {}) if isinstance(token_usage, dict) else {}
        if pass_usage:
            print("  Pass breakdown:")
            for pass_name, pass_stats in pass_usage.items():
                if not isinstance(pass_stats, dict):
                    continue
                print(
                    f"    {pass_name:<10} prompt={pass_stats.get('prompt_tokens')} | "
                    f"completion={pass_stats.get('completion_tokens')} | total={pass_stats.get('total_tokens')}"
                )

        print("\nGenerated JSON Result:")
        print(json.dumps(result, indent=2, ensure_ascii=False))
        print("-" * 80 + "\n")

    results_log.append({
        "file": filename,
        "time_s": processing_time,
        "tokens": total_tokens,
        "status": status,
        "policy": INFERENCE_POLICY,
    })
    print(f"{filename:<45} {str(processing_time):>8} {str(total_tokens):>7}  {status}")
    print()

    gc.collect()
    torch.cuda.empty_cache()

if results_log:
    results_df = pd.DataFrame(results_log)
    display(results_df)

if process_all:
    print(f"\nDone. {len(target_files)} image(s) processed. JSON outputs are in {GEN_DIR}")
else:
    print(f"\nRandom-sample inference complete for {len(target_files)} image(s). JSON outputs are in {GEN_DIR}")

## Step 6: OCR-Based BBox Refinement + Visualization

This step improves predicted bbox localization by matching each extracted text value back to OCR lines on the page and snapping the bbox to the matched OCR box.

Refined JSON files are saved to `GEN_DIR/refined_ocr` and then used for the overlay preview below.

In [ ]:
import json
import random
import re
import subprocess
import sys
from copy import deepcopy
from difflib import SequenceMatcher
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw, ImageFont

VIS_GROUND_TRUTH_DIR = Path(JSON_DIR)
VIS_OUTPUT_DIR = Path(GEN_DIR) / "overlay_output_with_fieldnames"
VIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATH_DIR = Path(IMAGE_DIR)
RAW_PREDICTION_PATH_DIR = Path(GEN_DIR)
REFINED_PREDICTION_PATH_DIR = RAW_PREDICTION_PATH_DIR / "refined_ocr"
REFINED_PREDICTION_PATH_DIR.mkdir(parents=True, exist_ok=True)

# OCR refinement switches
ENABLE_OCR_REFINEMENT = True
FORCE_RERUN_OCR_REFINEMENT = False
OCR_REFINEMENT_MIN_SCORE = 0.72

try:
    overlay_font = ImageFont.truetype("arial.ttf", 16)
except Exception:
    overlay_font = ImageFont.load_default()

RapidOCR = None
if ENABLE_OCR_REFINEMENT:
    try:
        from rapidocr_onnxruntime import RapidOCR
    except ImportError:
        print("Installing rapidocr-onnxruntime for OCR-based bbox refinement...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidocr-onnxruntime"], check=False)
        try:
            from rapidocr_onnxruntime import RapidOCR
        except Exception as exc:
            RapidOCR = None
            print(f"⚠ OCR refinement package unavailable: {exc}")
else:
    print("OCR refinement is disabled. Using raw predicted bboxes only.")

OCR_ENGINE = RapidOCR() if (ENABLE_OCR_REFINEMENT and RapidOCR is not None) else None
_OCR_CACHE = {}


def normalize_match_text(text):
    if text is None:
        return ""
    cleaned = str(text).strip().lower()
    cleaned = cleaned.replace("$", "")
    cleaned = cleaned.replace(",", "")
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned


def compact_match_text(text):
    return re.sub(r"[^a-z0-9./%:-]+", "", normalize_match_text(text))


def polygon_to_bbox(points):
    if not points or len(points) < 4:
        return None
    try:
        xs = [float(pt[0]) for pt in points]
        ys = [float(pt[1]) for pt in points]
    except Exception:
        return None
    return [round(min(xs), 1), round(min(ys), 1), round(max(xs), 1), round(max(ys), 1)]


def bbox_center(bbox):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return None
    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
    except Exception:
        return None
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def bbox_distance_score(box_a, box_b):
    center_a = bbox_center(box_a)
    center_b = bbox_center(box_b)
    if center_a is None or center_b is None:
        return 0.5
    dx = center_a[0] - center_b[0]
    dy = center_a[1] - center_b[1]
    dist = (dx ** 2 + dy ** 2) ** 0.5
    return max(0.0, 1.0 - (dist / 1500.0))


def load_ocr_entries(image_path):
    cache_key = str(image_path)
    if cache_key in _OCR_CACHE:
        return _OCR_CACHE[cache_key]

    entries = []
    if OCR_ENGINE is None or not Path(image_path).exists():
        _OCR_CACHE[cache_key] = entries
        return entries

    try:
        ocr_result, _ = OCR_ENGINE(str(image_path))
    except Exception as exc:
        print(f"⚠ OCR failed on {Path(image_path).name}: {exc}")
        _OCR_CACHE[cache_key] = entries
        return entries

    for item in ocr_result or []:
        if not isinstance(item, (list, tuple)) or len(item) < 3:
            continue
        points, text, score = item[0], item[1], item[2]
        bbox = polygon_to_bbox(points)
        if not text or bbox is None:
            continue
        try:
            score = float(score)
        except Exception:
            score = 0.0
        entries.append({
            "text": str(text),
            "norm": normalize_match_text(text),
            "compact": compact_match_text(text),
            "bbox": bbox,
            "score": score,
        })

    _OCR_CACHE[cache_key] = entries
    return entries


def find_best_ocr_match(target_text, ocr_entries, current_bbox=None):
    target_norm = normalize_match_text(target_text)
    target_compact = compact_match_text(target_text)
    if not target_compact:
        return None

    best_entry = None
    best_score = 0.0

    for entry in ocr_entries:
        norm = entry["norm"]
        compact = entry["compact"]
        if not compact:
            continue

        exact_score = 1.0 if target_compact == compact else 0.0
        contains_score = 0.0
        if target_compact in compact or compact in target_compact:
            contains_score = min(len(target_compact), len(compact)) / max(len(target_compact), len(compact))

        fuzzy_score = max(
            SequenceMatcher(None, target_norm, norm).ratio(),
            SequenceMatcher(None, target_compact, compact).ratio(),
        )
        text_score = max(exact_score, contains_score, fuzzy_score)
        location_score = bbox_distance_score(current_bbox, entry["bbox"])
        combined_score = (0.82 * text_score) + (0.18 * location_score)
        combined_score *= 0.9 + (0.1 * max(0.0, min(entry["score"], 1.0)))

        if combined_score > best_score:
            best_score = combined_score
            best_entry = entry

    min_required = OCR_REFINEMENT_MIN_SCORE if len(target_compact) >= 4 else 0.88
    if best_entry is None or best_score < min_required:
        return None

    return {
        "bbox": best_entry["bbox"],
        "ocr_text": best_entry["text"],
        "score": round(best_score, 4),
    }


def refine_docile_bboxes(pred_data, image_path):
    refined = deepcopy(pred_data)
    ocr_entries = load_ocr_entries(image_path)
    if not ocr_entries:
        return refined, {"candidate_boxes": 0, "refined_boxes": 0}

    candidate_boxes = 0
    refined_boxes = 0

    for section_name in ("field_extractions", "line_item_extractions"):
        for item in refined.get(section_name, []):
            text_value = item.get("text")
            if not text_value:
                continue

            candidate_boxes += 1
            original_bbox = item.get("bbox")
            match = find_best_ocr_match(text_value, ocr_entries, current_bbox=original_bbox)
            if match is None:
                continue

            if original_bbox != match["bbox"]:
                item["_bbox_model_raw"] = original_bbox
                item["bbox"] = match["bbox"]
                item["_ocr_match_text"] = match["ocr_text"]
                item["_ocr_match_score"] = match["score"]
                refined_boxes += 1

    meta = refined.setdefault("__metadata__", {})
    meta["ocr_bbox_refinement"] = {
        "enabled": ENABLE_OCR_REFINEMENT,
        "candidate_boxes": candidate_boxes,
        "refined_boxes": refined_boxes,
        "engine": "RapidOCR" if OCR_ENGINE is not None else "unavailable",
    }
    return refined, meta["ocr_bbox_refinement"]


def draw_bbox_and_label(draw, bbox, label, color, width=2):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return
    x1, y1, x2, y2 = bbox
    draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
    label_y = max(y1 - 18, 0)
    draw.text((x1, label_y), label, fill=color, font=overlay_font)


def collect_overlay_items(doc_json, prefix, color):
    items = []
    for entry in doc_json.get("field_extractions", []) + doc_json.get("line_item_extractions", []):
        bbox = entry.get("bbox")
        if not bbox or len(bbox) != 4:
            continue
        page = entry.get("page")
        field_type = entry.get("fieldtype", "unknown")
        line_item_id = entry.get("line_item_id")
        suffix = f"#{line_item_id}" if line_item_id is not None else ""
        items.append({
            "page": page,
            "bbox": bbox,
            "label": f"{prefix}:{field_type}{suffix}",
            "color": color,
        })
    return items


raw_prediction_files = sorted(RAW_PREDICTION_PATH_DIR.glob("*.json"))
if not raw_prediction_files:
    print(f"No prediction JSON files found in {RAW_PREDICTION_PATH_DIR}")
else:
    if not ENABLE_OCR_REFINEMENT:
        print("OCR refinement switch is OFF. Visualization will use raw predicted bboxes.")
    elif OCR_ENGINE is None:
        print("⚠ OCR engine unavailable, visualization will use raw predicted bboxes.")
    else:
        updated_files = 0
        total_candidates = 0
        total_refined = 0

        for pred_file in raw_prediction_files:
            image_file = IMAGE_PATH_DIR / f"{pred_file.stem}.png"
            out_file = REFINED_PREDICTION_PATH_DIR / pred_file.name

            if out_file.exists() and not FORCE_RERUN_OCR_REFINEMENT:
                continue
            if not image_file.exists():
                continue

            with open(pred_file, "r", encoding="utf-8") as f:
                pred_data = json.load(f)

            refined_data, stats = refine_docile_bboxes(pred_data, image_file)
            total_candidates += stats.get("candidate_boxes", 0)
            total_refined += stats.get("refined_boxes", 0)

            with open(out_file, "w", encoding="utf-8") as f:
                json.dump(refined_data, f, indent=2, ensure_ascii=False)
            updated_files += 1

        print(f"OCR refinement complete. Updated {updated_files} file(s).")
        print(f"Matched/refined boxes: {total_refined}/{total_candidates}")

    use_refined_dir = ENABLE_OCR_REFINEMENT and list(REFINED_PREDICTION_PATH_DIR.glob("*.json"))
    visualization_dir = REFINED_PREDICTION_PATH_DIR if use_refined_dir else RAW_PREDICTION_PATH_DIR
    prediction_files = sorted(visualization_dir.glob("*.json"))

    if "results_log" in locals() and results_log:
        latest_processed = results_log[-1]["file"]
        stem = Path(latest_processed).stem
        chosen_pred = next((p for p in prediction_files if p.stem == stem), None)
        if not chosen_pred:
            chosen_pred = random.choice(prediction_files)
    else:
        chosen_pred = random.choice(prediction_files)

    stem = chosen_pred.stem
    gt_file = VIS_GROUND_TRUTH_DIR / f"{stem}.json"
    image_file = IMAGE_PATH_DIR / f"{stem}.png"

    print(f"OCR refinement enabled: {ENABLE_OCR_REFINEMENT}")
    print(f"Using predictions from: {visualization_dir}")
    print(f"Selected file      : {stem}")
    print(f"Prediction JSON    : {chosen_pred}")
    print(f"Ground truth JSON  : {gt_file}")
    print(f"Image file         : {image_file}")

    if not image_file.exists():
        print("Matching image file not found.")
    elif not gt_file.exists():
        print("Matching ground-truth JSON file not found.")
    else:
        with open(chosen_pred, "r", encoding="utf-8") as f:
            pred_data = json.load(f)
        with open(gt_file, "r", encoding="utf-8") as f:
            gt_data = json.load(f)

        image = Image.open(image_file).convert("RGB")
        draw = ImageDraw.Draw(image)

        gt_items = collect_overlay_items(gt_data, "GT", (0, 200, 0))
        pred_items = collect_overlay_items(pred_data, "PRED", (255, 0, 0))

        for item in gt_items + pred_items:
            page = item.get("page")
            if page not in (0, 1):
                continue
            draw_bbox_and_label(draw, item["bbox"], item["label"], item["color"])

        out_file = VIS_OUTPUT_DIR / f"{stem}_overlay.png"
        image.save(out_file)
        print(f"Saved overlay: {out_file}")
        display(image)

## Step 7: Evaluate Predictions Against Ground Truth

This section compares predictions against the ground-truth DocILE JSON files and automatically prefers `GEN_DIR/refined_ocr` if the OCR bbox-refinement step has been run.

It matches the compact multi-pass flow by:

- handling metadata-only fields such as `language`, `document_type`, and `currency`
- preserving repeated tax-table rows instead of overwriting them
- evaluating OCR-refined outputs when available

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

EXPECTED_FIELDS = [
    "language", "document_type", "currency",
    "document_id", "date_due", "date_issue", "purchase_order_id", "terms",
    "tax_amount", "sender_vat_id", "tax_name", "amount_due", "amount_total_tax",
    "amount_total_base", "recipient_delivery_address", "recipient_delivery_name",
    "recipient_address", "recipient_name", "vendor_phone", "sender_address",
    "sender_name", "item_uom", "item_amount_total", "item_amount",
    "item_description", "item_code", "item_quantity",
]

METADATA_FIELDS = {"language", "document_type", "currency"}

GROUND_TRUTH_DIR = Path(JSON_DIR)
RAW_PREDICTION_DIR = Path(GEN_DIR)
REFINED_PREDICTION_DIR = RAW_PREDICTION_DIR / "refined_ocr"
PREDICTION_DIR = REFINED_PREDICTION_DIR if REFINED_PREDICTION_DIR.exists() and list(REFINED_PREDICTION_DIR.glob("*.json")) else RAW_PREDICTION_DIR
BBOX_IOU_THRESHOLD = 0.6


def safe_div(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def compute_prf(tp, fp, fn):
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    f1 = safe_div(2 * precision * recall, precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


def to_bbox_tuple(bbox):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return None
    try:
        return tuple(float(value) for value in bbox)
    except (TypeError, ValueError):
        return None


def bbox_iou(box_a, box_b):
    a = to_bbox_tuple(box_a)
    b = to_bbox_tuple(box_b)
    if a is None or b is None:
        return None

    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union_area = area_a + area_b - inter_area

    if union_area <= 0:
        return None
    return inter_area / union_area


def format_bbox(bbox):
    if bbox is None:
        return "[-]"
    parsed = to_bbox_tuple(bbox)
    if parsed is None:
        return str(bbox)
    return f"[{parsed[0]:.1f}, {parsed[1]:.1f}, {parsed[2]:.1f}, {parsed[3]:.1f}]"


def normalize_text_for_eval(text):
    if text is None:
        return ""
    cleaned = str(text).strip().lower()
    cleaned = cleaned.replace("$", "")
    cleaned = cleaned.replace(",", "")
    cleaned = " ".join(cleaned.split())
    return cleaned


def make_eval_entry(text=None, bbox=None, page=None, source=None):
    return {
        "text": text,
        "bbox": bbox,
        "page": page,
        "source": source,
    }


def sort_entry_key(entry):
    page = entry.get("page")
    try:
        page = int(page) if page is not None else 0
    except Exception:
        page = 0

    bbox = to_bbox_tuple(entry.get("bbox"))
    if bbox is None:
        bbox = (10**9, 10**9, 10**9, 10**9)

    return (page, bbox[1], bbox[0], normalize_text_for_eval(entry.get("text")))


def dedupe_and_sort_entries(entries):
    deduped = []
    seen = set()

    for entry in entries:
        dedupe_key = (
            normalize_text_for_eval(entry.get("text")),
            to_bbox_tuple(entry.get("bbox")),
            entry.get("page"),
        )
        if dedupe_key in seen:
            continue
        seen.add(dedupe_key)
        deduped.append(entry)

    return sorted(deduped, key=sort_entry_key)


def get_field_map(data):
    mapping = defaultdict(list)

    for item in data.get("field_extractions", []):
        if not isinstance(item, dict):
            continue
        field_type = item.get("fieldtype")
        if field_type:
            mapping[(field_type, None)].append(
                make_eval_entry(
                    text=item.get("text"),
                    bbox=item.get("bbox"),
                    page=item.get("page"),
                    source="field_extractions",
                )
            )

    for item in data.get("line_item_extractions", []):
        if not isinstance(item, dict):
            continue
        field_type = item.get("fieldtype")
        if field_type:
            mapping[(field_type, item.get("line_item_id"))].append(
                make_eval_entry(
                    text=item.get("text"),
                    bbox=item.get("bbox"),
                    page=item.get("page"),
                    source="line_item_extractions",
                )
            )

    metadata = data.get("metadata", {})
    if isinstance(metadata, dict):
        for field_type in METADATA_FIELDS:
            key = (field_type, None)
            if mapping.get(key):
                continue
            value = metadata.get(field_type)
            if value not in (None, "", [], {}):
                mapping[key].append(
                    make_eval_entry(text=value, bbox=None, page=0, source="metadata")
                )

    return {key: dedupe_and_sort_entries(entries) for key, entries in mapping.items()}


def to_float_or_none(value):
    try:
        return float(value)
    except Exception:
        return None


def extract_runtime_metadata(pred_data):
    metadata_block = pred_data.get("__metadata__", {})
    token_usage = pred_data.get("token_usage", {})
    ocr_info = metadata_block.get("ocr_bbox_refinement", {})
    pass_usage = token_usage.get("passes", {}) if isinstance(token_usage, dict) else {}
    return {
        "processing_time_s": to_float_or_none(pred_data.get("processing_time_s", metadata_block.get("processing_time", None))),
        "prompt_tokens": to_float_or_none(token_usage.get("prompt_tokens", metadata_block.get("prompt_tokens", None))),
        "completion_tokens": to_float_or_none(token_usage.get("completion_tokens", metadata_block.get("completion_tokens", None))),
        "total_tokens": to_float_or_none(token_usage.get("total_tokens", metadata_block.get("token_count", None))),
        "finish_reason": pred_data.get("finish_reason", metadata_block.get("finish_reason", "N/A")),
        "ocr_refined_boxes": int(ocr_info.get("refined_boxes", 0) or 0),
        "ocr_candidate_boxes": int(ocr_info.get("candidate_boxes", 0) or 0),
        "pass_names": ", ".join(pass_usage.keys()) if isinstance(pass_usage, dict) and pass_usage else "N/A",
    }


def sort_eval_key(item):
    field_type, line_item_id = item
    if line_item_id is None:
        return (-1, field_type)
    try:
        return (int(line_item_id), field_type)
    except Exception:
        return (999999, f"{line_item_id}_{field_type}")


def init_count_dict():
    return {"tp": 0, "fp": 0, "fn": 0, "correct": 0, "gt_total": 0}


def update_counts(counter, has_gt, has_pred, is_match):
    if has_gt:
        counter["gt_total"] += 1
    if has_gt and has_pred and is_match:
        counter["tp"] += 1
        counter["correct"] += 1
    elif has_gt and has_pred and not is_match:
        counter["fp"] += 1
        counter["fn"] += 1
    elif has_gt and not has_pred:
        counter["fn"] += 1
    elif has_pred and not has_gt:
        counter["fp"] += 1


def summarize_counter(counter):
    precision, recall, f1 = compute_prf(counter["tp"], counter["fp"], counter["fn"])
    accuracy = safe_div(counter["correct"], counter["gt_total"])
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": counter["tp"],
        "fp": counter["fp"],
        "fn": counter["fn"],
        "gt_total": counter["gt_total"],
    }


def evaluate_extractions(pred_dir, gt_dir, bbox_iou_threshold=0.6):
    pred_path = Path(pred_dir)
    gt_path = Path(gt_dir)

    if not pred_path.exists() or not gt_path.exists():
        print("Prediction or Ground Truth directory not found.")
        print(f"  Prediction dir : {pred_path}")
        print(f"  Ground truth dir: {gt_path}")
        return None

    overall_text = init_count_dict()
    overall_bbox = init_count_dict()
    field_counters = defaultdict(lambda: {"text": init_count_dict(), "bbox": init_count_dict()})
    file_summaries = []
    all_detail_rows = []

    for gt_file in sorted(gt_path.glob("*.json")):
        pred_file = pred_path / gt_file.name
        if not pred_file.exists():
            print(f"Skipping {gt_file.name}: prediction file not found")
            continue

        with open(gt_file, "r", encoding="utf-8") as f:
            gt_data = json.load(f)
        with open(pred_file, "r", encoding="utf-8") as f:
            pred_data = json.load(f)

        gt_fields = get_field_map(gt_data)
        pred_fields = get_field_map(pred_data)
        runtime_info = extract_runtime_metadata(pred_data)

        print(f"\n{'=' * 70}")
        print(f"Evaluating: {gt_file.name}")
        print(
            f"Time: {runtime_info['processing_time_s']}s | "
            f"Prompt: {runtime_info['prompt_tokens']} | "
            f"Completion: {runtime_info['completion_tokens']} | "
            f"Total: {runtime_info['total_tokens']} | "
            f"Finish: {runtime_info['finish_reason']} | "
            f"OCR refined: {runtime_info['ocr_refined_boxes']}/{runtime_info['ocr_candidate_boxes']} | "
            f"Passes: {runtime_info['pass_names']}"
        )
        print(f"{'=' * 70}")

        file_results = []
        file_text = init_count_dict()
        file_bbox = init_count_dict()

        all_keys = set(gt_fields.keys()).union(set(pred_fields.keys()))
        sorted_keys = sorted(all_keys, key=sort_eval_key)

        for field_type, line_item_id in sorted_keys:
            if field_type not in EXPECTED_FIELDS:
                continue

            gt_entries = gt_fields.get((field_type, line_item_id), [])
            pred_entries = pred_fields.get((field_type, line_item_id), [])
            max_count = max(len(gt_entries), len(pred_entries))

            for occurrence_idx in range(max_count):
                gt_entry = gt_entries[occurrence_idx] if occurrence_idx < len(gt_entries) else {}
                pred_entry = pred_entries[occurrence_idx] if occurrence_idx < len(pred_entries) else {}

                gt_text = gt_entry.get("text")
                pred_text = pred_entry.get("text")
                gt_bbox = gt_entry.get("bbox")
                pred_bbox = pred_entry.get("bbox")
                gt_page = gt_entry.get("page")
                pred_page = pred_entry.get("page")

                if gt_text is None and pred_text is None and gt_bbox is None and pred_bbox is None:
                    continue

                gt_clean = normalize_text_for_eval(gt_text)
                pred_clean = normalize_text_for_eval(pred_text)
                has_gt_text = gt_text not in (None, "", [], {})
                has_pred_text = pred_text not in (None, "", [], {})
                text_match = has_gt_text and has_pred_text and (gt_clean == pred_clean)

                if gt_page is None and pred_page is None:
                    page_match = True
                else:
                    page_match = gt_page == pred_page

                iou = bbox_iou(gt_bbox, pred_bbox)
                has_gt_bbox = gt_bbox is not None
                has_pred_bbox = pred_bbox is not None
                if not has_gt_bbox and not has_pred_bbox:
                    bbox_match = True
                elif has_gt_bbox and has_pred_bbox:
                    bbox_match = (iou is not None and iou >= bbox_iou_threshold and page_match)
                else:
                    bbox_match = False

                update_counts(file_text, has_gt_text, has_pred_text, text_match)
                update_counts(file_bbox, has_gt_bbox, has_pred_bbox, bbox_match)
                update_counts(overall_text, has_gt_text, has_pred_text, text_match)
                update_counts(overall_bbox, has_gt_bbox, has_pred_bbox, bbox_match)
                update_counts(field_counters[field_type]["text"], has_gt_text, has_pred_text, text_match)
                update_counts(field_counters[field_type]["bbox"], has_gt_bbox, has_pred_bbox, bbox_match)

                row = {
                    "File": gt_file.name,
                    "Field Type": field_type,
                    "Line ID": line_item_id if line_item_id is not None else "Header",
                    "Occurrence": occurrence_idx + 1 if max_count > 1 else "-",
                    "GT Source": gt_entry.get("source", "[-]"),
                    "Pred Source": pred_entry.get("source", "[-]"),
                    "Ground Truth": gt_clean if gt_text is not None else "[-]",
                    "Prediction": pred_clean if pred_text is not None else "[-]",
                    "Text Match": "✅" if text_match else "❌",
                    "GT Page": gt_page if gt_page is not None else "[-]",
                    "Pred Page": pred_page if pred_page is not None else "[-]",
                    "Page Match": "✅" if page_match else "❌",
                    "GT BBox": format_bbox(gt_bbox),
                    "Pred BBox": format_bbox(pred_bbox),
                    "BBox IoU": round(iou, 4) if iou is not None else "[-]",
                    "BBox Match": "✅" if bbox_match else "❌",
                }
                file_results.append(row)
                all_detail_rows.append(row)

        if file_results:
            display(pd.DataFrame(file_results))
        else:
            print("No expected fields found in this file.")

        file_text_summary = summarize_counter(file_text)
        file_bbox_summary = summarize_counter(file_bbox)
        file_summaries.append({
            "file": gt_file.name,
            "text_accuracy": file_text_summary["accuracy"],
            "text_precision": file_text_summary["precision"],
            "text_recall": file_text_summary["recall"],
            "text_f1": file_text_summary["f1"],
            "bbox_accuracy": file_bbox_summary["accuracy"],
            "bbox_precision": file_bbox_summary["precision"],
            "bbox_recall": file_bbox_summary["recall"],
            "bbox_f1": file_bbox_summary["f1"],
            "processing_time_s": runtime_info["processing_time_s"],
            "total_tokens": runtime_info["total_tokens"],
            "ocr_refined_boxes": runtime_info["ocr_refined_boxes"],
        })

    overall_text_summary = summarize_counter(overall_text)
    overall_bbox_summary = summarize_counter(overall_bbox)

    overall_summary = pd.DataFrame([
        {"metric": "accuracy", "text": overall_text_summary["accuracy"], "bbox": overall_bbox_summary["accuracy"]},
        {"metric": "precision", "text": overall_text_summary["precision"], "bbox": overall_bbox_summary["precision"]},
        {"metric": "recall", "text": overall_text_summary["recall"], "bbox": overall_bbox_summary["recall"]},
        {"metric": "f1", "text": overall_text_summary["f1"], "bbox": overall_bbox_summary["f1"]},
    ])

    field_rows = []
    for field_name, counters in sorted(field_counters.items()):
        text_summary = summarize_counter(counters["text"])
        bbox_summary = summarize_counter(counters["bbox"])
        field_rows.append({
            "field": field_name,
            "text_accuracy": text_summary["accuracy"],
            "text_precision": text_summary["precision"],
            "text_recall": text_summary["recall"],
            "text_f1": text_summary["f1"],
            "bbox_accuracy": bbox_summary["accuracy"],
            "bbox_precision": bbox_summary["precision"],
            "bbox_recall": bbox_summary["recall"],
            "bbox_f1": bbox_summary["f1"],
            "support": text_summary["gt_total"],
        })

    file_summary_df = pd.DataFrame(file_summaries)
    field_summary_df = pd.DataFrame(field_rows).sort_values(["text_f1", "support"], ascending=[False, False]) if field_rows else pd.DataFrame()
    detailed_df = pd.DataFrame(all_detail_rows)

    print("\n--- Overall Evaluation ---")
    display(overall_summary)
    if not file_summary_df.empty:
        display(file_summary_df.sort_values("text_f1", ascending=False).reset_index(drop=True))

    return {
        "overall_summary": overall_summary,
        "file_summary_df": file_summary_df,
        "field_summary_df": field_summary_df,
        "detailed_df": detailed_df,
        "overall_text": overall_text_summary,
        "overall_bbox": overall_bbox_summary,
    }


print(f"Prediction dir  : {PREDICTION_DIR}")
print(f"Ground truth dir: {GROUND_TRUTH_DIR}")
EVAL_RESULTS = evaluate_extractions(PREDICTION_DIR, GROUND_TRUTH_DIR, bbox_iou_threshold=BBOX_IOU_THRESHOLD)

## Step 8: Overall Evaluation Dashboard

This step turns the evaluation summary into charts so you can quickly inspect:

- overall text vs bbox metrics
- per-file F1 / accuracy trends
- runtime vs token cost
- field-level text F1

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from IPython.display import display

if "EVAL_RESULTS" not in globals() or EVAL_RESULTS is None:
    raise RuntimeError("Run Step 7 first to generate EVAL_RESULTS.")

overall_summary = EVAL_RESULTS["overall_summary"].copy()
file_summary_df = EVAL_RESULTS["file_summary_df"].copy()
field_summary_df = EVAL_RESULTS["field_summary_df"].copy()

print("Overall metrics summary:")
display(overall_summary.style.format({"text": "{:.3f}", "bbox": "{:.3f}"}).background_gradient(cmap="Blues", subset=["text", "bbox"]))

if not file_summary_df.empty:
    print("Per-file summary:")
    display(
        file_summary_df.sort_values("text_f1", ascending=False).reset_index(drop=True).style.format({
            "text_accuracy": "{:.3f}",
            "text_precision": "{:.3f}",
            "text_recall": "{:.3f}",
            "text_f1": "{:.3f}",
            "bbox_accuracy": "{:.3f}",
            "bbox_precision": "{:.3f}",
            "bbox_recall": "{:.3f}",
            "bbox_f1": "{:.3f}",
            "processing_time_s": "{:.2f}",
            "total_tokens": "{:.0f}",
        }).background_gradient(cmap="YlGnBu", subset=[
            "text_accuracy", "text_precision", "text_recall", "text_f1",
            "bbox_accuracy", "bbox_precision", "bbox_recall", "bbox_f1",
        ])
    )

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1) Overall summary heatmap
overall_heatmap = overall_summary.set_index("metric")[["text", "bbox"]].T
sns.heatmap(
    overall_heatmap,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Overall Text vs BBox Metrics")
axes[0, 0].set_xlabel("Metric")
axes[0, 0].set_ylabel("Category")

# 2) Per-file summary heatmap
if not file_summary_df.empty:
    heatmap_cols = [
        "text_accuracy", "text_precision", "text_recall", "text_f1",
        "bbox_accuracy", "bbox_precision", "bbox_recall", "bbox_f1",
    ]
    heatmap_df = file_summary_df.set_index("file")[heatmap_cols].sort_values("text_f1", ascending=False)
    sns.heatmap(
        heatmap_df,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        vmin=0,
        vmax=1,
        linewidths=0.5,
        ax=axes[0, 1],
        cbar=False,
    )
    axes[0, 1].set_title("Per-File Metrics Heatmap")
    axes[0, 1].set_xlabel("Metric")
    axes[0, 1].set_ylabel("File")
else:
    axes[0, 1].text(0.5, 0.5, "No per-file evaluation data", ha="center", va="center", transform=axes[0, 1].transAxes)
    axes[0, 1].set_title("Per-File Metrics Heatmap")

# 3) Per-file F1 comparison
if not file_summary_df.empty:
    ranked_files = file_summary_df.sort_values("text_f1", ascending=False)
    x_labels = ranked_files["file"].tolist()
    x_positions = range(len(x_labels))
    axes[1, 0].bar([x - 0.2 for x in x_positions], ranked_files["text_f1"], width=0.4, label="Text F1", color="#2ecc71")
    axes[1, 0].bar([x + 0.2 for x in x_positions], ranked_files["bbox_f1"], width=0.4, label="BBox F1", color="#3498db")
    axes[1, 0].set_xticks(list(x_positions))
    axes[1, 0].set_xticklabels(x_labels, rotation=30, ha="right")
    axes[1, 0].set_ylim(0, 1.05)
    axes[1, 0].set_title("Per-File F1 Comparison")
    axes[1, 0].legend()
else:
    axes[1, 0].text(0.5, 0.5, "No per-file evaluation data", ha="center", va="center", transform=axes[1, 0].transAxes)
    axes[1, 0].set_title("Per-File F1 Comparison")

# 4) Runtime vs total tokens
if not file_summary_df.empty and {"total_tokens", "processing_time_s"}.issubset(file_summary_df.columns):
    runtime_df = file_summary_df.copy()
    runtime_df["total_tokens"] = pd.to_numeric(runtime_df["total_tokens"], errors="coerce")
    runtime_df["processing_time_s"] = pd.to_numeric(runtime_df["processing_time_s"], errors="coerce")
    runtime_df = runtime_df.dropna(subset=["total_tokens", "processing_time_s"])
    if not runtime_df.empty:
        sns.scatterplot(
            data=runtime_df,
            x="total_tokens",
            y="processing_time_s",
            size="ocr_refined_boxes",
            hue="text_f1",
            palette="magma",
            sizes=(80, 300),
            ax=axes[1, 1],
        )
        axes[1, 1].set_title("Runtime vs Total Tokens")
    else:
        axes[1, 1].text(0.5, 0.5, "No runtime/token data", ha="center", va="center", transform=axes[1, 1].transAxes)
        axes[1, 1].set_title("Runtime vs Total Tokens")
else:
    axes[1, 1].text(0.5, 0.5, "No runtime/token data", ha="center", va="center", transform=axes[1, 1].transAxes)
    axes[1, 1].set_title("Runtime vs Total Tokens")

plt.tight_layout(rect=[0, 0, 1, 0.98])
dashboard_path = Path(GEN_DIR) / "evaluation_dashboard.png"
plt.savefig(dashboard_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved evaluation dashboard: {dashboard_path}")

if not field_summary_df.empty:
    plt.figure(figsize=(10, 6))
    top_fields = field_summary_df.sort_values(["text_f1", "support"], ascending=[False, False]).head(10)
    sns.barplot(data=top_fields, x="text_f1", y="field", palette="crest")
    plt.xlim(0, 1.05)
    plt.title("Top 10 Fields by Text F1")
    plt.xlabel("Text F1")
    plt.ylabel("Field")
    plt.tight_layout()
    field_chart_path = Path(GEN_DIR) / "field_text_f1_chart.png"
    plt.savefig(field_chart_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved field chart: {field_chart_path}")

## Step 9: DEMO

Run inference on `invoice_sample.png` using **2 models**:
1. Base Model (`Qwen3-VL-8B`)
2. Finetuned Model (`final_adapter`)

Both will have their predictions visualized with OCR-based bbox refinement for side-by-side comparison.

In [ ]:
# --- STEP 9 CLEANUP (Uncomment to use) ---
# To clear the specific JSON for THIS image:
# if os.path.exists(demo_gen_json_path): os.remove(demo_gen_json_path); print("✓ Cleared specific JSON.")

# To clear ALL generated JSONs in the working folder:
import glob
for f in glob.glob(os.path.join(GEN_DIR, "*.json")): os.remove(f); print(f"✓ Removed {os.path.basename(f)}")

# To clear ALL refined OCR results:
if os.path.exists(REFINED_PREDICTION_PATH_DIR): shutil.rmtree(REFINED_PREDICTION_PATH_DIR); os.makedirs(REFINED_PREDICTION_PATH_DIR); print("✓ Cleared refined OCR results.")

# To clear ALL overlay images:
import shutil
if os.path.exists(VIS_OUTPUT_DIR): shutil.rmtree(VIS_OUTPUT_DIR); os.makedirs(VIS_OUTPUT_DIR); print("✓ Cleared overlay images.")


In [ ]:
import os
import gc
import torch
import json
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

# Input configuration (Explicitly absolute paths)
demo_image_name = "invoice_sample.png"
demo_image_path = os.path.abspath(os.path.join(IMAGE_DIR, demo_image_name))
demo_gt_path = os.path.abspath(os.path.join(JSON_DIR, os.path.splitext(demo_image_name)[0] + ".json"))
demo_gen_json_path = os.path.abspath(os.path.join(GEN_DIR, os.path.splitext(demo_image_name)[0] + ".json"))
base_json_path = os.path.abspath(os.path.join(GEN_DIR, f"{os.path.splitext(demo_image_name)[0]}_base.json"))
ft_json_path   = os.path.abspath(os.path.join(GEN_DIR, f"{os.path.splitext(demo_image_name)[0]}_ft.json"))

if not os.path.exists(demo_image_path):
    print(f"Image not found: {demo_image_path}")
else:
    # Ensure Ground Truth is loaded if available
    gt_data = None
    if os.path.exists(demo_gt_path):
        with open(demo_gt_path, "r", encoding="utf-8") as f:
            gt_data = json.load(f)
        print("✓ Loaded ground truth for overlay.")
    else:
        print("⚠ Ground truth JSON not found for this file.")

    def run_demo_pass(title_prefix, color, cache_path=None, force_rerun=True):
        print(f"\n{'='*60}\nRunning Pass: {title_prefix}\n{'='*60}")
    
        result = None

        # ---- Cache handling ----
        if not force_rerun and cache_path and os.path.exists(cache_path):
            print(f"📦 Using cached result: {cache_path}")
            with open(cache_path, "r", encoding="utf-8") as f:
                result = json.load(f)

        # ---- Inference ----
        if result is None:
            gc.collect()
            torch.cuda.empty_cache()

            result = run_vision_inference(demo_image_path, decode_mode="greedy")

            if "error" in result:
                print(f"❌ Error during inference: {result['error']}")
                return None

            print("✓ Inference complete (fresh run)")

            if cache_path:
                with open(cache_path, "w", encoding="utf-8") as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)
                print(f"♻️ Saved JSON: {cache_path}")

        # ---- OCR Refinement ----
        try:
            refined_data, stats = refine_docile_bboxes(result, Path(demo_image_path))
            print(f"✓ Refined {stats.get('refined_boxes', 0)} boxes using OCR.")
        except Exception as e:
            print(f"⚠ OCR Refinement failed: {e}. Using raw bboxes.")
            refined_data = result

        # ---- Visualization ----
        image = Image.open(demo_image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
    
        pred_items = collect_overlay_items(refined_data, "PRED", color)

        gt_items = []
        if gt_data:
            gt_items = collect_overlay_items(gt_data, "GT", (0, 200, 0))
    
        for item in gt_items + pred_items:
            if item.get("page") in (0, 1):
                draw_bbox_and_label(draw, item["bbox"], item["label"], item["color"])

        # ---- Save image ----
        mode_tag = "base" if "BASE" in title_prefix.upper() else "ft"

        save_name = f"{os.path.splitext(demo_image_name)[0]}_{mode_tag}_overlay.png"
        save_path = os.path.join(VIS_OUTPUT_DIR, save_name)

        image.save(save_path)
        print(f"♻️ Saved overlay image: {save_path}")

        print(f"--- Visualization: {title_prefix} ---")
        display(image)

        return refined_data

In [ ]:
# --- EXECUTION ---

# BASE
print("Switching context to BASE MODEL (No LoRA)...")

try:
    if hasattr(model, "disable_adapter"):
        with model.disable_adapter():
            base_result = run_demo_pass(
                "BASE MODEL",
                (255, 0, 0),
                cache_path=base_json_path,
                force_rerun=True
            )
    else:
        print("Model doesn't have disable_adapter.")
        base_result = None
except Exception as e:
    print(f"Error toggling adapter: {e}")
    base_result = None

In [ ]:
# FINETUNED
print("\nSwitching context to FINETUNED MODEL (final_adapter)...")

ft_result = run_demo_pass(
    "FINETUNED MODEL",
    (255, 0, 0),
    cache_path=ft_json_path,
    force_rerun=True
)

print("\nStandalone Demo complete.")